# 03-short. Colab용 PPG 대안모델 + Optuna 빠른 실험

기존 `03_ppg_alternative_models_optuna.ipynb`는 상세 검사용입니다.  
이 파일은 **Colab에서 짧게 여러 모델을 비교**하기 위한 버전입니다.

- 기본값은 `LIMIT_SUBJECTS = 12`, `FOLDS = 3`, `OPTUNA_TRIALS = 5`입니다.
- 전체 실험은 `LIMIT_SUBJECTS = None`, `FOLDS = 5`, `OPTUNA_TRIALS = 20+`로 바꾸세요.
- LightGBM은 `DEVICE = "auto"`로 GPU 가능 시 GPU를 먼저 씁니다.


## 0. Colab/repo 환경 준비


In [ ]:
# Colab에서 이 노트북만 열어도 동작하도록 repo와 의존성을 준비합니다.
# 로컬 repo/Jupyter에서는 아무것도 설치하지 않고 현재 repo root만 찾습니다.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tlxhtls/llamac_research.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: Path | None = None) -> Path:
    """현재 위치에서 위로 올라가며 llamac_research repo root를 찾습니다."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "llamac_research").exists():
            return candidate
    raise RuntimeError(f"repo root를 찾지 못했습니다: {start}")


if IN_COLAB:
    root = Path("/content/llamac_research")
    if not (root / "src" / "llamac_research").exists():
        print("Colab에 repo를 clone합니다...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    os.chdir(root)
    print("Colab 의존성을 설치합니다. 처음 한 번은 1~3분 걸릴 수 있습니다...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".",
            "lightgbm>=4.5",
            "optuna>=4.0",
            "scikit-learn>=1.5",
            "xgboost>=2.1",
        ],
        check=True,
    )
    ROOT = root
else:
    ROOT = find_repo_root()
    os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT = {ROOT}")


## 1. 실험 설정


In [ ]:
LIMIT_SUBJECTS = 12
FOLDS = 3
SEED = 42
WORKERS = 2
DEVICE = "auto"
OPTUNA_TRIALS = 5

REBUILD_FEATURES = False
RERUN_BASELINES = False
RERUN_OPTUNA = False

FEATURE_DIR = ROOT / "data" / "processed" / "short_colab"
RESULT_DIR = ROOT / "artifacts" / "short_colab" / "ppg_alternatives"
OPTUNA_DIR = ROOT / "artifacts" / "short_colab" / "optuna"
FEATURE_PPG = FEATURE_DIR / "features_ppg_short.parquet"
FEATURE_PPG_RICH = FEATURE_DIR / "features_ppg_rich_short.parquet"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
print(f"LIMIT_SUBJECTS={LIMIT_SUBJECTS}, FOLDS={FOLDS}, DEVICE={DEVICE}, OPTUNA_TRIALS={OPTUNA_TRIALS}")


## 2. 데이터셋 다운로드와 압축해제


In [ ]:
# Figshare DOI 10.6084/m9.figshare.28748696.v6 에서 zip을 내려받고 압축을 풉니다.
# 빠른 Colab 데모는 LIMIT_SUBJECTS명만 받습니다. 전체 재현은 LIMIT_SUBJECTS = None 으로 바꾸세요.
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import polars as pl

from scripts.download_llamac import (
    FigshareFile,
    build_dataset_index,
    download_one,
    extract_archives,
    fetch_metadata,
    natural_key,
    write_manifest,
)

RAW_DIR = ROOT / "data" / "raw"
EXTRACTED_DIR = ROOT / "data" / "extracted"
PROCESSED_DIR = ROOT / "data" / "processed"
INDEX_PATH = PROCESSED_DIR / "dataset_index.csv"
SKIP_HEAVY = os.environ.get("LLAMAC_SKIP_HEAVY") == "1"

ARTICLE_ID = "28748696"
VERSION = "6"
DOWNLOAD_WORKERS = 4
MAX_ZIPS = LIMIT_SUBJECTS  # None이면 전체 zip 파일을 다운로드합니다.

if INDEX_PATH.exists():
    print(f"이미 준비된 index가 있어서 다운로드를 건너뜁니다: {INDEX_PATH}")
elif SKIP_HEAVY:
    print("LLAMAC_SKIP_HEAVY=1 이므로 다운로드를 건너뜁니다.")
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    metadata = fetch_metadata(ARTICLE_ID, VERSION)
    files = [FigshareFile.from_api(item) for item in metadata["files"]]
    zip_files = sorted([f for f in files if f.name.lower().endswith(".zip")], key=lambda f: natural_key(f.name))
    selected = zip_files[:MAX_ZIPS] if MAX_ZIPS is not None else zip_files
    write_manifest(metadata, RAW_DIR, selected)

    print(f"다운로드 대상 zip: {len(selected)}개")
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_one, f, RAW_DIR, False, 3, 120, False) for f in selected]
        for i, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            print(f"[{i}/{len(selected)}] {result['name']}: {result['status']}")

    extract_archives(RAW_DIR, selected, EXTRACTED_DIR, force=False)
    build_dataset_index(EXTRACTED_DIR, INDEX_PATH)

if INDEX_PATH.exists():
    index = pl.read_csv(INDEX_PATH)
    print(index.select(pl.len().alias("files"), pl.col("participant_id").n_unique().alias("participants")))
else:
    index = None


## 3. backend 확인


In [ ]:
from llamac_research.device import select_lightgbm_device, select_torch_device

lgb_backend = select_lightgbm_device(DEVICE)
torch_backend = select_torch_device(DEVICE)

print("[LightGBM]")
print(lgb_backend.to_dict())
print("\n[Torch 참고용]")
print(torch_backend.to_dict())


## 4. PPG / rich-PPG feature 생성 또는 재사용


In [ ]:
from llamac_research.features import build_feature_table, read_feature_table


def ensure_features(mode: str, path: Path) -> pl.DataFrame:
    if path.exists() and not REBUILD_FEATURES:
        print(f"재사용: {path}")
        return read_feature_table(path)
    if SKIP_HEAVY:
        print(f"검증 모드라 feature 생성을 건너뜁니다: {path}")
        return pl.DataFrame()
    frame, summary = build_feature_table(
        EXTRACTED_DIR,
        mode=mode,
        limit_subjects=LIMIT_SUBJECTS,
        workers=WORKERS,
        output_path=path,
    )
    print(summary.to_dict())
    return frame

ppg = ensure_features("ppg", FEATURE_PPG)
ppg_rich = ensure_features("ppg_rich", FEATURE_PPG_RICH)

print("ppg:", ppg.shape)
print("ppg_rich:", ppg_rich.shape)


## 5. 여러 대안모델을 같은 split으로 빠르게 비교


In [ ]:
# XGBoost는 현재 공통 harness의 label 인코딩과 맞지 않아 기본 비교에서 제외합니다.
# GPU 가속 확인은 LightGBM backend 출력으로 확인합니다.
import json

from llamac_research.modeling import ExperimentConfig, run_cross_validated_experiment, save_experiment_result
from llamac_research.metrics import metrics_summary_line

BASELINE_MODELS = ["logistic_regression", "lightgbm", "extra_trees", "hist_gradient_boosting"]


def run_or_load_model(model_name: str, feature_path: Path, feature_label: str) -> dict | None:
    out = RESULT_DIR / f"{model_name}_{feature_label}_grouped.json"
    if out.exists() and not RERUN_BASELINES:
        print(f"[{model_name}/{feature_label}] 저장된 결과 재사용")
        return json.loads(out.read_text())
    if SKIP_HEAVY:
        print(f"[{model_name}/{feature_label}] 검증 모드라 학습을 건너뜁니다.")
        return None
    cfg = ExperimentConfig(
        feature_path=str(feature_path),
        model_name=model_name,
        feature_set="ppg",  # rich 파일도 PPG 계열 column만 선택합니다.
        target="reported",
        split_strategy="grouped",
        n_splits=FOLDS,
        seed=SEED,
        device=DEVICE,
    )
    result = run_cross_validated_experiment(cfg)
    save_experiment_result(result, out)
    print(f"[{model_name}/{feature_label}] {metrics_summary_line(result['metrics'])}")
    return result

baseline_results = {}
for model in BASELINE_MODELS:
    baseline_results[f"{model}_ppg"] = run_or_load_model(model, FEATURE_PPG, "ppg")

# rich PPG는 feature가 조금 더 많아서, 대표적으로 logistic + LightGBM만 빠르게 봅니다.
for model in ["logistic_regression", "lightgbm"]:
    baseline_results[f"{model}_ppg_rich"] = run_or_load_model(model, FEATURE_PPG_RICH, "ppg_rich")


## 6. Optuna 짧은 탐색


In [ ]:
# 짧은 노트북이라 기본 trial은 작게 둡니다. 논문용 탐색은 OPTUNA_TRIALS를 20~100+로 늘리세요.
from llamac_research.tuning import TuningConfig, run_tuning

optuna_summary_path = OPTUNA_DIR / "llamac_ppg_reported_summary.json"
if optuna_summary_path.exists() and not RERUN_OPTUNA:
    print(f"Optuna summary 재사용: {optuna_summary_path}")
    optuna_summary = json.loads(optuna_summary_path.read_text())
elif SKIP_HEAVY:
    print("검증 모드라 Optuna를 건너뜁니다.")
    optuna_summary = None
else:
    tune_cfg = TuningConfig(
        feature_path=str(FEATURE_PPG),
        models=["logistic_regression", "lightgbm"],
        feature_set="ppg",
        target="reported",
        split_strategy="grouped",
        n_splits=FOLDS,
        seed=SEED,
        device=DEVICE,
        n_trials=OPTUNA_TRIALS,
        metric="macro_f1",
        output_dir=str(OPTUNA_DIR),
        study_prefix="llamac",
    )
    optuna_summary = run_tuning(tune_cfg)


## 7. 결과 표와 그래프


In [ ]:
import matplotlib.pyplot as plt

rows = []
for name, result in baseline_results.items():
    if result is None:
        continue
    m = result["metrics"]
    rows.append({
        "run": name,
        "kind": "fixed",
        "backend": result["backend"]["selected"],
        "top1": round(m["top1_accuracy"], 4),
        "macro_f1": round(m["macro_f1"], 4),
        "balanced_acc": round(m["balanced_accuracy"], 4),
        "features": result["data"]["candidate_features"],
    })

if optuna_summary:
    for study in optuna_summary.get("studies", []):
        m = study["locked_metrics"]
        rows.append({
            "run": f"optuna_{study['model_name']}",
            "kind": "optuna_locked",
            "backend": "auto",
            "top1": round(m["top1_accuracy"], 4),
            "macro_f1": round(m["macro_f1"], 4),
            "balanced_acc": round(m["balanced_accuracy"], 4),
            "features": None,
        })

comparison = pl.DataFrame(rows).sort("macro_f1", descending=True) if rows else pl.DataFrame()
display(comparison)

if comparison.height:
    plot_df = comparison.sort("macro_f1")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(plot_df["run"].to_list(), plot_df["macro_f1"].to_numpy())
    ax.set_xlabel("macro F1")
    ax.set_title("PPG 대안모델 비교")
    ax.set_xlim(0, 1)
    fig.tight_layout()
    plt.show()
